# Train TinyLlama HelpSteer2 Adapters

This notebook trains five independent TinyLlama LoRA specialists for helpfulness, correctness, coherence, complexity, and verbosity. It uses the central experiment configuration and provides graceful stop controls for long Colab runs.

## 1. Clone or update the repository

In [ ]:
%cd /content
import os
import shutil

repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"

if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis

## 2. Check the GPU

In [ ]:
!nvidia-smi

## 3. Install dependencies

The pandas and NumPy versions are pinned for compatibility with the standard Colab environment. TinyLlama uses standard LoRA training without 4-bit quantization.

In [ ]:
!pip install -q -U "pandas==2.2.2" "numpy<2.1" transformers datasets peft accelerate pyyaml tensorboard

Restart the runtime once if Colab asks for it after installation. Then rerun the repository cell before continuing.

## 4. Show important files

In [ ]:
!pwd
!ls
!ls configs
!ls scripts
!ls src

## 5. Validate the config and inspect HelpSteer2

In [ ]:
!python scripts/validate_tinyllama_helpsteer2_config.py
!python scripts/inspect_helpsteer2_dataset.py --split "train[:10000]"

## 6. Start TensorBoard

Open the **Scalars** view to inspect training and evaluation loss against `global_step`. New points appear as training writes event logs.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/tensorboard/tinyllama_helpsteer2

## 7. Run a single-attribute smoke test

Remove stale stop files before starting a new run.

In [ ]:
!rm -f STOP_CURRENT_ADAPTER STOP_TRAINING
!python scripts/train_tinyllama_helpsteer2_adapters.py --attributes helpfulness --split "train[:20]" --num_epochs 1 --use_tensorboard

## 8. Check the saved helpfulness adapter

In [ ]:
!python scripts/check_tinyllama_helpsteer2_adapters.py --attributes helpfulness

## 9. Start one adapter training run

Train one adapter per command. Start `helpfulness`, `correctness`, `coherence`, `complexity`, and `verbosity` manually, changing only `--attributes` for each run. For helpfulness, the command is:

```bash
python scripts/train_tinyllama_helpsteer2_adapters.py --attributes helpfulness --split "train[:10000]" --eval_split "train[10000:11000]" --num_epochs 50 --batch_size 8 --max_length 1024 --learning_rate 1e-4 --logging_steps 10 --eval_steps 100 --save_steps 500 --use_tensorboard
```

The cell below starts it as a background process so the stop controls remain available. Progress is written to `/content/tinyllama_helpsteer2_training.log`.

In [ ]:
!rm -f STOP_CURRENT_ADAPTER STOP_TRAINING
!nohup python -u scripts/train_tinyllama_helpsteer2_adapters.py --attributes helpfulness --split "train[:10000]" --eval_split "train[10000:11000]" --num_epochs 50 --batch_size 8 --max_length 1024 --learning_rate 1e-4 --logging_steps 10 --eval_steps 100 --save_steps 500 --use_tensorboard > /content/tinyllama_helpsteer2_training.log 2>&1 &
!sleep 3
!tail -n 30 /content/tinyllama_helpsteer2_training.log

## 10. Stop the current adapter

Use this to continue only until the next `save_steps` boundary, save the current adapter/checkpoint, remove `STOP_CURRENT_ADAPTER`, and stop this script run.

In [ ]:
!touch STOP_CURRENT_ADAPTER

## 11. Stop this training run

Use this to continue only until the next `save_steps` boundary, save the current adapter/checkpoint, and stop this script run. `STOP_TRAINING` is not removed automatically. A smaller `save_steps` value reduces the maximum wait before stopping.

In [ ]:
!touch STOP_TRAINING

## 12. Optional stop buttons

In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

status = widgets.Output()
stop_current_button = widgets.Button(
    description="Stop at next save step",
    button_style="warning",
)
stop_all_button = widgets.Button(
    description="Stop run at next save step",
    button_style="danger",
)

def request_current_stop(_):
    Path("STOP_CURRENT_ADAPTER").touch()
    with status:
        status.clear_output()
        print("STOP_CURRENT_ADAPTER created. Waiting for the next save step.")

def request_full_stop(_):
    Path("STOP_TRAINING").touch()
    with status:
        status.clear_output()
        print("STOP_TRAINING created. Waiting for the next save step.")

stop_current_button.on_click(request_current_stop)
stop_all_button.on_click(request_full_stop)
display(widgets.VBox([stop_current_button, stop_all_button, status]))

## 13. Inspect progress and logs

Rerun this cell whenever you want to inspect recent progress.

In [ ]:
!tail -n 50 /content/tinyllama_helpsteer2_training.log 2>/dev/null || true
!ls results/tinyllama_helpsteer2_training_logs 2>/dev/null || true
!ls results/tensorboard/tinyllama_helpsteer2 2>/dev/null || true

## 14. Check all trained adapters

In [ ]:
!python scripts/check_tinyllama_helpsteer2_adapters.py

## 15. Create a local adapter backup

The zip file is a local backup. Download it from Colab and keep it out of GitHub.

In [ ]:
!zip -r tinyllama_helpsteer2_adapters.zip adapters/tinyllama-helpsteer2-*-adapter/

## 16. Git safety check

Adapters, checkpoints, `.safetensors`, `.bin`, zip files, and model weights are generated artifacts and must not be committed.

In [ ]:
!git status